# Import libraries

## Biopython gives us access to PubMed's Entrez API

In [3]:
### Bio = Biopython library
### Entrez = PubMed's API tool inside it

from Bio import Entrez
import json

# Always tell PubMed who you are (required by their API)
Entrez.email = "tpriya27@gmail.com"

print("Libraries imported successfully!")


Libraries imported successfully!


### Block 1: Data Ingestion 

### Search PubMed for Alzheimer's papers
Entrez.efetch() to retrieve full bibliographic information or abstracts. This automated approach is especially valuable when you need to analyze hundreds or thousands of papers programmatically.



In [4]:
# We're asking PubMed: "give me paper IDs 
# related to Alzheimer's early detection"
#db="pubmed"     → which database to search
#term="..."      → your search query
#retmax=10       → maximum 10 results
#paper_ids       → list of PubMed ID numbers
                  

search_handle = Entrez.esearch(
    db="pubmed",
    term="Alzheimer's early detection biomarkers",
    retmax=100
)

search_results = Entrez.read(search_handle)
search_handle.close()

paper_ids = search_results["IdList"]

print("Found paper IDs:", paper_ids)
print("Total found:", len(paper_ids))

Found paper IDs: ['42045900', '42045435', '42042246', '42041411', '42037712', '42035900', '42034728', '42031363', '42031084', '42024117', '42024084', '42023593', '42022738', '42013966', '42013560', '42013009', '42009747', '42009666', '42008697', '42008415', '42008257', '42007415', '42000570', '41999930', '41997807', '41997556', '41994995', '41988865', '41983466', '41979952', '41977873', '41977515', '41977405', '41976093', '41974986', '41971641', '41970527', '41956234', '41953934', '41953170', '41952858', '41949058', '41948713', '41948540', '41948065', '41944116', '41941099', '41940849', '41940846', '41936289', '41935672', '41933850', '41929951', '41929949', '41929947', '41929942', '41928210', '41925947', '41921464', '41918502', '41918278', '41918201', '41917601', '41915112', '41914594', '41913058', '41911308', '41905188', '41904574', '41903861', '41897293', '41896724', '41896267', '41895180', '41894949', '41892906', '41892599', '41892053', '41891033', '41887009', '41884150', '41883703'

### Fetch The Actual Papers

####  IDs to get the real titles and abstracts:

In [5]:
fetch_handle = Entrez.efetch(
    db="pubmed",
    id= paper_ids,
    rettype="abstract",
    retmode="xml"
)

papers = Entrez.read(fetch_handle)
fetch_handle.close()

print("Papers fetched successfully!")
print("Number of papers:", len(papers["PubmedArticle"]))

Papers fetched successfully!
Number of papers: 100


## Extract The Useful Information

####  This is the data we'll later convert to embeddings

In [21]:
#articles        → list of all 10 papers
#enumerate()     → gives index + item together
#try/except      → some papers have no abstract
                  #we handle that gracefully
#[:200]          → show only first 200 characters
                  #abstracts are very long!

In [6]:


articles = papers["PubmedArticle"]

for i, article in enumerate(articles):
    
    # Extract title
    title = article["MedlineCitation"]["Article"]["ArticleTitle"]
    
    # Extract abstract (some papers may not have one)
    try:
        abstract = article["MedlineCitation"]["Article"]["Abstract"]["AbstractText"][0]
    except KeyError:
        abstract = "No abstract available"
    
    print(f"Paper {i+1}:")
    print(f"Title: {title}")
    print(f"Abstract: {str(abstract)[:200]}...")
    print("-" * 50)

Paper 1:
Title: Peripheral blood biomarkers RCAN1, Clusterin, RAGE, and malondialdehyde for early diagnosis and progression of Alzheimer's disease.
Abstract: No abstract available...
--------------------------------------------------
Paper 2:
Title: Alzheimer's disease-like brain pattern biomarker: capturing risks and predicting disease onset.
Abstract: Preventing Alzheimer's disease (AD) requires early-warning biomarkers. We developed a Regional Vulnerability Index (RVI) that quantifies individual brain similarity to AD patients' expected brain defi...
--------------------------------------------------
Paper 3:
Title: Explainable Patient-Level Cognitive Impairment Screening via Temporal, Semantic, and Psycholinguistic Multimodal AI.
Abstract: Early diagnosis of cognitive decline is vital for timely treatment of mild cognitive impairment (MCI) and Alzheimer's disease (AD), yet standard clinical assessments often miss subtle longitudinal lan...
------------------------------------------

Real, cutting-edge Alzheimer's research papers fetched live from PubMed! 
- ✅ "Tears as a window to Alzheimer's disease: biomarkers for early detection"
- ✅ "Early Diagnosis of Alzheimer's: Machine Learning Analysis"
- ✅ "Plasma GFAP and P-tau217 with imaging ATN markers"
These are 2026 papers — fresher than any AI's training data!

## Save The Data 

In [7]:
# Cell 5: Structure and save the data as JSON
# This becomes our raw data for the RAG pipeline

import json

structured_papers = []

for article in articles:
    
    title = article["MedlineCitation"]["Article"]["ArticleTitle"]
    
    try:
        abstract = article["MedlineCitation"]["Article"]["Abstract"]["AbstractText"][0]
    except KeyError:
        abstract = "No abstract available"
    
    # Extract PubMed ID
    pmid = str(article["MedlineCitation"]["PMID"])
    
    structured_papers.append({
        "pmid": pmid,
        "title": str(title),
        "abstract": str(abstract),
        "source": "PubMed"
    })

# Save to data folder
with open("../data/raw/alzheimer_papers.json", "w") as f:   
    json.dump(structured_papers, f, indent=2)

print(f"Saved {len(structured_papers)} papers to alzheimer_papers.json")
print("\nSample paper:")
print(json.dumps(structured_papers[0], indent=2))

Saved 100 papers to alzheimer_papers.json

Sample paper:
{
  "pmid": "42045900",
  "title": "Peripheral blood biomarkers RCAN1, Clusterin, RAGE, and malondialdehyde for early diagnosis and progression of Alzheimer's disease.",
  "abstract": "No abstract available",
  "source": "PubMed"
}
